### Load RDF Data in Python

In [21]:
from rdflib import Graph

# Load RDF
g = Graph()
g.parse(r"C:\Users\Osama Haider\Downloads\documents.ttl", format="ttl")

<Graph identifier=N9dc03bef69904b9a8ffb1bb61f6d153b (<class 'rdflib.graph.Graph'>)>

### Compute BERT Embeddings

In [22]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract document contents and compute embeddings
docs = []
doc_texts = []

for s, p, o in g.triples((None, None, None)):
    if str(p).endswith("content"):
        docs.append(s)       # document URI
        doc_texts.append(str(o))

# Check if we got documents
print("Documents found:", len(doc_texts))
print(doc_texts)

# Compute embeddings
doc_embeddings = model.encode(doc_texts)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents found: 3
['This article talks about AI development laptops with powerful GPUs and CPUs.', 'Guide to hardware and software tools for AI research.', 'This article covers budget-friendly laptops suitable for students.']


### Implement Semantic Search Function

In [26]:
from rdflib import Namespace
from sklearn.metrics.pairwise import cosine_similarity

EX = Namespace("http://example.org/")

def semantic_search(query, top_k=3):
    # Compute embedding for the query
    query_embedding = model.encode([query])
    
    # Compute cosine similarity with documents
    sims = cosine_similarity(query_embedding, doc_embeddings)[0]
    
    # Get top K most similar documents
    top_idx = sims.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_idx:
        doc_uri = docs[idx]
        # Correctly fetch the title from RDF
        title = g.value(subject=doc_uri, predicate=EX.title)
        results.append((str(title), sims[idx]))
    return results

### Interactive Query in Notebook

In [27]:
query = input("Enter your search query: ")
results = semantic_search(query)

print("\nTop Results:")
for title, score in results:
    print(f"{title} (Similarity: {score:.2f})")

Enter your search query:  Which laptops are best for AI development?



Top Results:
Best Laptops for AI (Similarity: 0.72)
Top AI Tools and Hardware (Similarity: 0.58)
Affordable Laptops for Students (Similarity: 0.46)
